# Setup (run every session)

Kaggle and Colab reset between sessions, so run these cells top to bottom each time.

On Kaggle: turn on **Internet** and pick the **GPU T4 x2** accelerator in the notebook settings.

In [ ]:
import os, sys

REPO_URL = "https://github.com/shoimya/Is-the-Judge-the-Bottleneck.git"
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
WORK = "/content" if IN_COLAB else "/kaggle/working" if IN_KAGGLE else os.getcwd()
REPO = os.path.join(WORK, "Is-the-Judge-the-Bottleneck")
print("Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local", sys.version)

In [ ]:
# 1. Clone the repo, or pull the latest if it's already here
import socket
try:
    socket.gethostbyname("github.com")
except OSError:
    raise SystemExit("No internet. On Kaggle: verify your phone number (kaggle.com > Settings), "
                     "then switch Internet on in the notebook's settings panel and run all again.")

if os.path.isdir(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone {REPO_URL} {REPO}
assert os.path.isdir(REPO), "Clone failed: see the git error above."
%cd {REPO}

In [ ]:
# 2. Install pinned requirements
!pip install -q -r requirements.txt

In [ ]:
# 3. Keep runs/ somewhere that survives the session
#    Kaggle: /kaggle/working/runs (kept as notebook output when you Save Version)
#    Colab:  Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_TARGET = "/content/drive/MyDrive/Is-the-Judge-the-Bottleneck/runs"
elif IN_KAGGLE:
    RUNS_TARGET = "/kaggle/working/runs"
else:
    RUNS_TARGET = None

if RUNS_TARGET:
    os.makedirs(RUNS_TARGET, exist_ok=True)
    if not os.path.islink("runs"):
        !rm -rf runs
        os.symlink(RUNS_TARGET, "runs")
    print("runs/ ->", os.path.realpath("runs"))

In [ ]:
# 4. Setup test
!python run.py --check

# 5. BM25 index over HotpotQA's Wikipedia (T03)

Run the setup cells first, then run 5a and wait for it to finish successfully
before running 5c. The download is about 1.55 GB; build time and memory use
depend on the session.

The builder prints the absolute index path from the project configuration
(default: `<repo>/data/wiki/bm25_index`). It creates the question sets if they
are missing and runs the Pilot recall check after the build. The recall CSV
is saved to `results/pilot/bm25_recall.csv`. Use Save Version to preserve the
notebook output under `/kaggle/working`.

In [ ]:
# 5a. Build the index and run the Pilot sanity check
from pathlib import Path
import subprocess
import sys

# Use the repository from setup, even if the notebook's working directory changed.
repo_dir = Path(REPO).resolve()
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from pipeline import REPO_ROOT, load_config
from pipeline.retriever import index_dir

print('Index destination:', index_dir(), flush=True)

def run_pipeline(*args):
    subprocess.run(
        [sys.executable, '-u', '-m', *args],
        cwd=REPO_ROOT,
        check=True,
    )

pilot_path = REPO_ROOT / load_config()['paths']['data_dir'] / 'hotpotqa' / 'pilot.jsonl'
if not pilot_path.is_file():
    run_pipeline('pipeline.dataset')

# A failed command stops this cell, preserving the original build error.
run_pipeline('pipeline.retriever', 'build')
run_pipeline('pipeline.retriever', 'sanity')
print('Build and Pilot check completed. You can now run 5c.')

# 5b. Share the index with the Mac (private Kaggle Dataset)

Every machine should load the same saved index with memory mapping. Cell 5c
uses `pipeline.retriever.index_dir()` to locate it, checks the required files,
and copies it into the export folder. Changing the notebook's working
directory does not change the source path. If the check fails, inspect the
output of 5a and resolve its build error before exporting.

1. After 5a succeeds, run 5c to prepare `/kaggle/working/hotpotqa_bm25/`.
2. Save the notebook output, then create a private Kaggle Dataset containing
   the export folder's `bm25_index/` and `dataset-metadata.json`.
3. On the Mac: `kaggle datasets download -d <user>/<dataset> -p data/wiki --unzip`

Keep a copy of the complete exported index on Drive as well.

In [ ]:
# 5c. Pack the index for the private Dataset (run after 5a)
from pathlib import Path
import json
import shutil
import sys

repo_dir = Path(REPO).resolve()
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from pipeline.retriever import index_dir

SOURCE = index_dir().resolve()
EXPORT = Path(WORK).resolve() / 'hotpotqa_bm25'
print('Index source:', SOURCE)
print('Export destination:', EXPORT)

required_files = (
    'params.index.json',
    'vocab.index.json',
    'data.csc.index.npy',
    'indices.csc.index.npy',
    'indptr.csc.index.npy',
    'corpus.jsonl',
)
missing = [
    name for name in required_files
    if not (SOURCE / name).is_file() or (SOURCE / name).stat().st_size == 0
]
if missing:
    raise FileNotFoundError(
        f'No complete saved index at {SOURCE}. Missing or empty: {missing}. '
        'Run cell 5a and resolve any build error before running 5c. '
        'The downloaded Wikipedia archive alone is not a BM25 index.'
    )

# Validate before touching the export; reruns do not delete existing files.
EXPORT.mkdir(parents=True, exist_ok=True)
shutil.copytree(SOURCE, EXPORT / 'bm25_index', dirs_exist_ok=True)
metadata = {
    'title': 'HotpotQA 2017 Wikipedia BM25 index (bm25s)',
    'licenses': [{'name': 'CC-BY-SA-4.0'}],
}
(EXPORT / 'dataset-metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)
print('Export ready:', EXPORT)
for path in sorted((EXPORT / 'bm25_index').iterdir()):
    print(path.name)

# 6. Model backend (T04)

Kaggle reports come from vLLM only. 6a installs the GPU stack, 6b points the
config at the model and runs the 3 test prompts through `pipeline.llm`, and
6c measures throughput (paste the printed row into NOTES.md > Throughput).

One engine serves one T4 (tp=1). The main runs (T11) will run one process per
T4 (data parallel) with `CUDA_VISIBLE_DEVICES=0/1` instead of a single engine.

In [ ]:
# 6a. GPU stack (Kaggle only: vLLM does not install on the Mac)
# If the image refuses this pin, change requirements-gpu.txt to vllm==0.28.0.
!pip install -q -r requirements-gpu.txt

In [ ]:
# 6b. Point the config at vLLM and run the 3 test prompts (T04 done-when:
#     same prompts on both backends; logprobs present so P("yes") is readable)
from pathlib import Path
import subprocess
import sys
import yaml

repo_dir = Path(REPO).resolve()
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from pipeline import REPO_ROOT

config_path = REPO_ROOT / 'config.yaml'
config = yaml.safe_load(config_path.read_text())
config['llm']['backend'] = 'vllm'
# fp16 fits one T4; the 8B AWQ (T10) runs one copy per T4 instead.
config['models'] = {role: 'Qwen/Qwen3-4B-Instruct-2507'
                    for role in ('judge', 'rewriter', 'answerer')}
config_path.write_text(yaml.safe_dump(config))

# The first call loads the engine (a couple of minutes); then 3 completions.
subprocess.run([sys.executable, '-u', '-m', 'pipeline.llm'],
               cwd=REPO_ROOT, check=True)

In [ ]:
# 6c. Throughput: 20 Pilot questions through a fake 3-Round loop (T04).
#     Paste the printed markdown row into NOTES.md > Throughput.
subprocess.run([sys.executable, '-u', '-m', 'pipeline.throughput'],
               cwd=REPO_ROOT, check=True)

# 7. Role parse-rate check (T06)

Done-when: on 20 Pilot questions, each role's output parses at least 95% of
the time, and the Judge's p_yes is between 0 and 1 for every call. Run 6b
first (it points the config at vLLM). The checker retrieves the top-2
paragraphs as stand-in Evidence, calls the Judge and Answerer on them, and
the Rewriter on the Judge's steer signal.

In [ ]:
# 7. Run the parse-rate check (20 Pilot questions). Each role is called once
#    per question; expect >= 95% parsed per role and p_yes in [0, 1].
subprocess.run([sys.executable, '-u', '-m', 'pipeline.roles', '--questions', '20'],
               cwd=REPO_ROOT, check=True)

# 8. The loop, all seven settings (T08)

Done-when: every setting runs end-to-end on 10 Pilot questions and writes a
valid run log; Condition D reaches all Gold paragraphs at least as often as
Condition A; a setting that dies halfway resumes cleanly (restart without
`--fresh` skips logged questions). Run 6b first (config at vLLM) and have the
BM25 index in place (cell 5a / the private Dataset). Expect this to take most
of the session — it is the bulk of the GPU time.

In [ ]:
# 8a. Run all seven settings, 10 Pilot questions each. Each run writes its own
#     runs/<run_id>_<setting>/ folder and is resumable on restart.
for setting in ('A', 'B', 'C', 'D', 'single-turn', 'always-loop', 'closed-book'):
    print('=' * 20, setting, '=' * 20)
    subprocess.run([sys.executable, '-u', 'run.py', '--setting', setting,
                    '--set', 'pilot', '--questions', '10'],
                   cwd=REPO_ROOT, check=True)

In [ ]:
# 8b. T08 sanity: with oracle stop+steer (D), all Gold paragraphs are reached
#     at least as often as with the LLM judge (A).
from run import latest_run
from evaluate import gold_coverage, load_run

runs_dir = REPO_ROOT / 'runs'
a = load_run(latest_run(runs_dir, 'A', 'pilot'))
d = load_run(latest_run(runs_dir, 'D', 'pilot'))
print('gold coverage  A:', gold_coverage(a))
print('gold coverage  D:', gold_coverage(d))
assert gold_coverage(d) >= gold_coverage(a), 'oracle wiring sanity failed'
print('OK: D covers at least as much as A')